In [ ]:
# ============================================================
# OCR-BASED WHATSAPP SCREENSHOT ANALYSIS
# No training data needed - uses your existing text dataset!
# ============================================================

!pip install pytesseract pillow opencv-python --quiet
!apt-get update -qq && apt-get install -y tesseract-ocr -qq

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pytesseract
from PIL import Image
import cv2
import numpy as np
from google.colab import files

# STEP 1: Train on your existing text data (you already have this!)
df = pd.read_csv("./cyberbullying_data_labelled.csv")

X = df['cleaned_text'].astype(str)
y = df['oh_label'].astype(int)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')
X_tfidf = vectorizer.fit_transform(X)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_tfidf, y)

print(f"✅ Model trained on {len(df)} text samples!")

# STEP 2: Extract text from ANY screenshot
def extract_from_screenshot(image_path):
    """OCR to get text from WhatsApp screenshot"""
    img = Image.open(image_path)
    
    # Convert to array for preprocessing
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    
    # Improve OCR accuracy
    denoised = cv2.fastNlMeansDenoising(gray)
    _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # OCR
    text = pytesseract.image_to_string(thresh)
    return text

# STEP 3: Analyze extracted text
def analyze_screenshot(image_path):
    """Complete pipeline: Image → OCR → Text Analysis"""
    print(f"📸 Processing: {image_path}")
    
    # Extract text
    chat_text = extract_from_screenshot(image_path)
    
    if not chat_text.strip():
        print("❌ No text found in image")
        return None
    
    print(f"📝 Extracted {len(chat_text)} characters")
    print("-" * 50)
    print(chat_text[:500])  # Show first 500 chars
    print("-" * 50)
    
    # Split into messages and analyze each
    lines = [l.strip() for l in chat_text.split('\n') if l.strip()]
    
    bullying_count = 0
    flagged_messages = []
    
    for line in lines:
        if len(line) < 5:  # Skip short lines (timestamps, etc.)
            continue
            
        # Predict
        line_tfidf = vectorizer.transform([line.lower()])
        pred = model.predict(line_tfidf)[0]
        prob = model.predict_proba(line_tfidf)[0][1] * 100
        
        if pred == 1:
            bullying_count += 1
            flagged_messages.append((line, prob))
    
    # Results
    total_messages = len([l for l in lines if len(l) >= 5])
    bullying_ratio = (bullying_count / total_messages * 100) if total_messages > 0 else 0
    
    print(f"\n📊 RESULTS:")
    print(f"   Messages analyzed: {total_messages}")
    print(f"   Bullying messages: {bullying_count}")
    print(f"   Bullying ratio: {bullying_ratio:.1f}%")
    
    if bullying_ratio > 30:
        print(f"   🚨 VERDICT: CYBERBULLYING DETECTED")
    elif bullying_ratio > 10:
        print(f"   ⚠️ VERDICT: SUSPICIOUS")
    else:
        print(f"   ✅ VERDICT: SAFE")
    
    if flagged_messages:
        print(f"\n🚩 FLAGGED MESSAGES:")
        for msg, conf in sorted(flagged_messages, key=lambda x: x[1], reverse=True)[:5]:
            print(f"   • {conf:.1f}%: \"{msg[:60]}...\"")
    
    return bullying_ratio

# STEP 4: Upload and analyze
print("=" * 60)
print("📤 UPLOAD WHATSAPP SCREENSHOT")
print("=" * 60)

uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    with open(filename, 'wb') as f:
        f.write(uploaded[filename])
    
    # Analyze!
    analyze_screenshot(filename)